# Edge and boundary representation

Classify offline limiter and balanced double-null fixtures, then inspect X-points, named plasma-wall gaps, separatrix balance, and Fourier reconstruction metadata. Absolute X-point coordinates remain distinct from `dRsep`.

In [ ]:
import numpy as np
from vaft.data.equilibrium import Contour, EquilibriumConvention, EquilibriumData, Topology
from vaft.process.equilibrium import derive_boundary_representation, extract_flux_surface_contours

def fixture(double_null=False):
    r=np.linspace(0.5,1.5,121); z=np.linspace(-0.75,0.75,151); rm,zm=np.meshgrid(r,z,indexing="ij")
    if double_null:
        a=0.65; psi=(rm-1)**2+zm**2-zm**4/a**2; boundary=a**2/4
        raw=extract_flux_surface_contours(psi,r,z,0,boundary,[1.0])[1.0]
        rb,zb=max(raw,key=lambda pair: pair[0].size)
    else:
        psi=((rm-1)/0.35)**2+(zm/0.5)**2; boundary=1.0; theta=np.linspace(0,2*np.pi,361,endpoint=False); rb=1+0.35*np.cos(theta); zb=0.5*np.sin(theta)
    # A limited plasma is bounded by wall contact; a diverted one is not.
    theta=np.linspace(0,2*np.pi,361,endpoint=False)
    wall=Contour(1+0.42*np.cos(theta),0.62*np.sin(theta)) if double_null else Contour(1+0.35*np.cos(theta),0.55*np.sin(theta))
    return EquilibriumData(r=r,z=z,psi=psi,psi_axis=0,psi_boundary=boundary,magnetic_axis=(1,0),lcfs=Contour(rb,zb),limiter=wall,convention=EquilibriumConvention(11,(11,),False,False,1,1,1,"fixture"))

limited=derive_boundary_representation(fixture(False),fourier_modes=12)
# flux_tolerance is left unset so the X-point flux window is derived from
# the grid resolution and each saddle's own curvature.
double=derive_boundary_representation(fixture(True),fourier_modes=12)
print("limited:",limited.topology,"| wall contact %.4g m"%limited.wall_contact_distance.value)
print("  gaps:",[round(g.distance.value,4) for g in limited.gaps])
print("  saddles:",len(limited.x_points),"active:",sum(x.active for x in limited.x_points))
print("double-null:",double.topology,[(x.r,x.z,x.active) for x in double.x_points],"dRsep=",double.d_r_sep.value)
assert limited.topology is Topology.LIMITED and not limited.topology.is_diverted
assert double.topology is Topology.DOUBLE_NULL and double.topology.is_diverted

In [ ]:
print("dRsep definition:",double.d_r_sep.definition)
print("Fourier coefficient names:",sorted(double.fourier_coefficients))
print("Fourier RMS [m]:",double.fourier_reconstruction_error.value)
print("Strike-point diagnostic:",double.reason or f"{len(double.strike_points)} strike points")
print("X-point flux window (psi_n):",double.provenance.tolerances["xpoint_flux_psi_n"])